# Data Collection
## ESPN College Football Win Probability — Calibration Study

This notebook drives the full data collection pipeline:
1. Collect game IDs (schedule) for each season
2. Fetch play-by-play + win probability for each game
3. Optionally fetch Vegas lines from the College Football Data API
4. Process raw JSON → clean Parquet files

Run the cells sequentially. Collection can be paused and resumed safely — already-fetched games are skipped.

In [ ]:
import sys
sys.path.insert(0, "..")

import asyncio
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s")

# Paths
SCHEDULES_DIR = Path("../data/raw/schedules")
GAMES_DIR     = Path("../data/raw/games")
LINES_DIR     = Path("../data/raw/lines")
PROCESSED_DIR = Path("../data/processed")

# Seasons to collect — edit this list as needed
SEASONS = list(range(2015, 2026))

## Step 1: Collect Game Schedules (Game IDs)

Iterates every game date for FBS (groups=80) and FCS (groups=81) and saves
`data/raw/schedules/{season}.json`. Already-collected seasons are skipped.

In [ ]:
from cfb_calibration.scrapers import collect_schedules

# DRY_RUN=True just prints what would be fetched without making requests
DRY_RUN = False

schedules = await collect_schedules(SEASONS, SCHEDULES_DIR, dry_run=DRY_RUN)

if not DRY_RUN:
    total = sum(len(v) for v in schedules.values())
    print(f"\nTotal games in schedule: {total:,}")
    for season, games in sorted(schedules.items()):
        fbs = sum(1 for g in games if g.get("group") == 80)
        fcs = sum(1 for g in games if g.get("group") == 81)
        print(f"  {season}: {len(games):,} total  ({fbs:,} FBS, {fcs:,} FCS)")

## Step 2: Collect Play-by-Play + Win Probability

For each game ID, fetches the ESPN summary JSON and saves it as `data/raw/games/{season}/{game_id}.json.gz`.

**This step takes several hours for full history. It can be interrupted and resumed.**

In [ ]:
from cfb_calibration.scrapers import collect_all_games

all_statuses = await collect_all_games(schedules, GAMES_DIR)

# Summary
for season, statuses in sorted(all_statuses.items()):
    saved   = sum(1 for s in statuses.values() if s == "saved")
    skipped = sum(1 for s in statuses.values() if s == "skipped")
    errors  = sum(1 for s in statuses.values() if s == "error")
    print(f"{season}: saved={saved:,}  skipped={skipped:,}  errors={errors:,}")

## Step 3: Vegas Lines (Optional)

Requires a free API key from https://collegefootballdata.com/key

Add `CFBD_API_KEY=your_key_here` to a `.env` file in the project root, then run this cell.

In [ ]:
from cfb_calibration.scrapers import collect_lines

for season in SEASONS:
    collect_lines(season, LINES_DIR)

## Step 4: Build Processed Parquet Files

Parses all raw JSON into `data/processed/plays.parquet` and `data/processed/games.parquet`.

In [ ]:
from cfb_calibration.processing.pipeline import build_dataset

games_df, plays_df = build_dataset(
    raw_games_dir=GAMES_DIR,
    schedules_dir=SCHEDULES_DIR,
    lines_dir=LINES_DIR,
    output_dir=PROCESSED_DIR,
)

print(f"Games:  {len(games_df):,} rows, {games_df.shape[1]} columns")
print(f"Plays:  {len(plays_df):,} rows, {plays_df.shape[1]} columns")
print(f"\nMemory usage:")
print(f"  plays_df: {plays_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nSample play:")
plays_df.sample(1).T